# DermaXAI: tái lập kết quả trên Derm7pt

Notebook này lấy code từ GitHub (đúng commit của báo cáo tổng kết), nối dữ liệu và checkpoint từ phần Input, rồi chạy theo `MODE`.

**Chuẩn bị**
1. *Add Input*: một dataset chứa Derm7pt (thư mục có `images/` và `meta/`, tức `release_v0`) và dataset `dermaxai-checkpoints`.
2. *Settings*: Accelerator = GPU, Internet = On.
3. Chọn `MODE` ở ô dưới rồi *Run All*.

| `MODE` | Làm gì | Có chấm test không |
|---|---|---|
| `check` | Kiểm tra GPU, dữ liệu (413/203/395 ca) và hash SHA-256 của checkpoint so với `test_plan.json` | Không |
| `reproduce_test` | Chấm lại tập test bằng checkpoint và ngưỡng đã khóa, in bảng 5 metric và so với kết quả đã lưu | Có |
| `train_one` | Train lại một mô hình với một seed (chỉ train/validation, 50 epoch) và so với kết quả validation đã lưu | Không |

Kết quả chạy trên GPU CUDA có thể lệch nhẹ so với kết quả gốc chạy trên Apple MPS.

In [ ]:
MODE = "check"            # "check" | "reproduce_test" | "train_one"

REPO_URL = "https://github.com/LeNhatHuy219/DermaXAI.git"
COMMIT = "3ce666ac349526a545d4b1ef57bcfd2700446a36"   # bản ứng với báo cáo tổng kết

TRAIN_MODEL = "hard_cbm"  # dùng khi MODE = "train_one": "blackbox" | "soft_cbm" | "hard_cbm" | "ecbm"
TRAIN_SEED = 42

## 1. Lấy code

In [ ]:
import glob, hashlib, json, os, subprocess, sys

WORK = "/kaggle/working"
REPO = "/tmp/DermaXAI"   # ngoài /kaggle/working để Kaggle không lưu repo và ảnh làm output

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

if not os.path.isdir(REPO):
    sh(f"git clone -q {REPO_URL} {REPO}")
sh(f"git -C {REPO} checkout -q {COMMIT}")
try:
    import joblib, matplotlib, pandas, PIL, seaborn, sklearn, torch, torchvision, tqdm
except ImportError:
    sh(f"pip install -q -r {REPO}/requirements.txt")
os.chdir(REPO)
sys.path.insert(0, REPO)

## 2. Nối dữ liệu và checkpoint

In [ ]:
def find_dir(match):
    for root, dirs, _ in os.walk("/kaggle/input", followlinks=True):
        if match(root, dirs):
            return root
    return None

data_root = find_dir(lambda root, dirs: "images" in dirs and "meta" in dirs)
ckpt_root = find_dir(lambda root, dirs: {"ecbm", "hard_cbm", "soft_cbm", "efficientnet_b0"} <= set(dirs))
print("Derm7pt:   ", data_root)
print("Checkpoint:", ckpt_root)
assert data_root, "Chưa thấy Derm7pt trong Input (cần thư mục có images/ và meta/)"

os.makedirs("Derm7pt", exist_ok=True)
if not os.path.exists("Derm7pt/release_v0"):
    os.symlink(data_root, "Derm7pt/release_v0")
if ckpt_root:
    os.makedirs("checkpoints", exist_ok=True)
    if not os.path.exists("checkpoints/derm7pt"):
        os.symlink(ckpt_root, "checkpoints/derm7pt")

## 3. Kiểm tra môi trường, dữ liệu và checkpoint

In [ ]:
import pandas as pd
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có (sẽ chạy CPU, rất chậm)")

manifest = pd.read_csv("data/manifest.csv")
missing = [p for p in manifest["derm_path_resolved"] if not os.path.exists(p)]
print("Số ca theo split:", manifest["split"].value_counts().to_dict())
print("Ảnh thiếu:", len(missing))
assert not missing, missing[:5]

plan = json.load(open("results/derm7pt/test_plan.json"))
if ckpt_root:
    for name, entry in plan["models"].items():
        folder = entry["cohort"].replace("results/", "checkpoints/", 1)
        for seed, expected in entry["checkpoints"].items():
            path = f"{folder}/seed{seed}.pth"
            actual = hashlib.sha256(open(path, "rb").read()).hexdigest()
            print(f"{name:16s} seed {seed:>4s}: {'khớp' if actual == expected else 'LỆCH'}")
            assert actual == expected, path
else:
    print("Không có dataset checkpoint: chỉ chạy được MODE = 'train_one'.")

## 4. Bảng 5 metric chính

In [ ]:
import numpy as np
from src.metrics import compute_metrics

SEEDS = ("42", "123", "2026")
CBM_COHORTS = {
    "Black-box": "results/derm7pt/efficientnet_b0/linear_e50_headlr0.001",
    "Soft CBM": "results/derm7pt/soft_cbm/efficientnet_b0_mlp128_e50_headlr0.001",
    "Hard CBM": "results/derm7pt/hard_cbm/efficientnet_b0_mlp128_e50_headlr0.001",
}
ECBM_MODES = {"ECBM λ=1 (chính)": "ecbm_full", "ECBM λ paper (0.01)": "ecbm_paper", "ECBM chỉ concept (phụ)": "ecbm_concept_only"}

def _row(blocks):
    values = {"Acc %": [], "BAcc %": [], "Macro F1": [], "ROC-AUC": [], "Concept acc %": []}
    for block in blocks:
        p = block["test_predictions"]
        m = compute_metrics([x["y_true"] for x in p], [x["y_pred"] for x in p], [x["y_prob"] for x in p])
        values["Acc %"].append(100 * m["accuracy"])
        values["BAcc %"].append(100 * m["balanced_accuracy"])
        values["Macro F1"].append(m["f1_macro"])
        values["ROC-AUC"].append(m["roc_auc"])
        concepts = block.get("test_concept_metrics")
        if concepts:
            values["Concept acc %"].append(100 * concepts["overall_concept_accuracy"])
    return {k: (np.mean(v), np.std(v, ddof=1)) if v else None for k, v in values.items()}

def main_table():
    rows = {}
    for name, folder in CBM_COHORTS.items():
        rows[name] = _row([json.load(open(f"{folder}/seed{s}.json")) for s in SEEDS])
    ecbm = [json.load(open(f"results/derm7pt/ecbm/test_modes/seed{s}.json")) for s in SEEDS]
    for name, key in ECBM_MODES.items():
        rows[name] = _row([run["modes"][key] for run in ecbm])
    return rows

def show(rows):
    fmt = lambda cell, col: "—" if cell is None else (
        f"{cell[0]:.4f} ± {cell[1]:.4f}" if col in ("Macro F1", "ROC-AUC") else f"{cell[0]:.2f} ± {cell[1]:.2f}")
    display(pd.DataFrame({name: {col: fmt(cell, col) for col, cell in row.items()} for name, row in rows.items()}).T)

saved = main_table()
print("Kết quả đã lưu trong repo (chưa chạy lại gì):")
show(saved)

## 5. `reproduce_test`: chấm lại tập test bằng checkpoint đã khóa

In [ ]:
if MODE == "reproduce_test":
    assert ckpt_root, "Cần dataset checkpoint"
    for runner in ("run_blackbox.py", "run_soft_cbm.py", "run_hard_cbm.py"):
        for seed in SEEDS:
            sh(f"python -B experiments/{runner} --mode test --seed {seed} --overwrite")
    sh("python -B experiments/run_ecbm_modes.py --stage test --overwrite")

    rerun = main_table()
    print("Kết quả chạy lại trên Kaggle:")
    show(rerun)
    diffs = {name: {col: abs(rerun[name][col][0] - saved[name][col][0])
                    for col in saved[name] if saved[name][col] is not None} for name in saved}
    print("Chênh lệch tuyệt đối lớn nhất so với kết quả đã lưu (mean):")
    display(pd.DataFrame(diffs).T.round(4))
else:
    print("Bỏ qua (MODE khác 'reproduce_test').")

## 6. `train_one`: train lại một mô hình, một seed (không chấm test)

In [ ]:
if MODE == "train_one":
    runners = {"blackbox": "run_blackbox.py", "soft_cbm": "run_soft_cbm.py",
               "hard_cbm": "run_hard_cbm.py", "ecbm": "run_ecbm.py"}
    stored = {"blackbox": CBM_COHORTS["Black-box"], "soft_cbm": CBM_COHORTS["Soft CBM"],
              "hard_cbm": CBM_COHORTS["Hard CBM"], "ecbm": "results/derm7pt/ecbm/efficientnet_b0_unified_exact_e50"}
    out = f"{WORK}/train_{TRAIN_MODEL}_seed{TRAIN_SEED}"
    os.makedirs(out, exist_ok=True)
    sh(f"python -B experiments/{runners[TRAIN_MODEL]} --seed {TRAIN_SEED} --skip_test "
       f"--checkpoint_path {out}/model.pth --results_path {out}/result.json --overwrite")

    def summary(result):
        return {"best epoch": result["best_epoch"],
                "validation Macro F1@0.5": round(result["validation_metrics_at_0.5"]["f1_macro"], 4),
                "validation Macro F1 (ngưỡng đã chọn)": round(result["validation_metrics"]["f1_macro"], 4),
                "validation concept acc %": round(100 * result["validation_concept_metrics"]["overall_concept_accuracy"], 2)}
    display(pd.DataFrame({"Kết quả gốc": summary(json.load(open(f"{stored[TRAIN_MODEL]}/seed{TRAIN_SEED}.json"))),
                          "Train lại trên Kaggle": summary(json.load(open(f"{out}/result.json")))}))
else:
    print("Bỏ qua (MODE khác 'train_one').")